In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-ridho-model'  # ckpt = 4000
exp_name = 'friction-walking-terrain1-kp2000kd50-kpkdrand-26-norand'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-28'  # min_ankle_height 弊害
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-8'  #  暫定１位
# exp_name = 'bp000-walking'
# exp_name = 'friction-walking-fractal-norand'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []
dof_pos_data = []
dof_vel_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
env_cfg["episode_length_s"] = 60.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
env_cfg['dt'] = 0.01
env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.07
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 60.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.08,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {}}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()
dof_pos = env.dof_pos[0].cpu().numpy()
dof_vel = env.dof_vel[0].cpu().numpy()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())
dof_pos_data.append(dof_pos)
dof_vel_data.append(dof_vel)

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)

    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[ 0.0282, -0.1058,  0.2325, -0.0409, -0.7817, -0.0324,  0.0613, -0.0198,
          0.3866, -0.2790, -0.7475,  0.0842]], device='cuda:0')
Scaled actions :  tensor([[ 0.0282, -0.1058,  0.2325, -0.0409, -0.7817, -0.0324,  0.0613, -0.0198,
          0.3866, -0.2790, -0.7475,  0.0842]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-3.6060e-06, -7.2420e-03, -1.6779e-06,  5.3713e-10, -3.8003e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.0648e-07,
         -5.6872e-08, -1.4913e-04,  3.7074e-04, -1.9264e-04,  8.3635e-07,
          1.3636e-08,  2.6061e-08, -1.4901e-04,  3.7062e-04, -1.9264e-04,
         -2.8088e-08, -5.3240e-06, -2.8436e-06, -7.4566e-03,  1.8541e-02,
         -9.6340e-03,  4.1817e-05,  6.8178e-07,  1.3030e-06, -7.4525e-03,
          1.8535e-02, -9.6340e-03, -1.4044e-06,  2.8184e-02, -1.0580e-01,
          2.3251e-01, -4.0947e-02, -7.8169e-01, -3.2383e-02,  6.1343e-02,
         -1.9835e-02,  3.8656e-01, -2.7905e-01, -7.4747e-01,  8.4234e-02]],
       device='cuda:0')
torques: [-5.80341944e-17  5.74598420e-16  3.12308246e-06  6.53296839e-06
  1.36950939e-06  3.84571231e-17  9.06233176e-17  4.24078100e-16
  3.12308246e-06  6.53296839e-06  1.36950939e-06 -4.88784221e-17]
dof_pos: [-1.0647950e-07 -5.6872231e-08 -8.0014914e-01  1.6003708e+00
 -8.0019265e-01  8.3634905

In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)

    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[-0.0109,  0.2395, -0.3302, -0.0538, -0.6919, -0.0286,  0.0652,  0.6694,
          0.1044, -0.1323, -0.9079,  0.0219]], device='cuda:0')
Scaled actions :  tensor([[-0.0109,  0.2395, -0.3302, -0.0538, -0.6919, -0.0286,  0.0652,  0.6694,
          0.1044, -0.1323, -0.9079,  0.0219]], device='cuda:0')
obs :  tensor([[ 0.2852, -0.7348, -0.2029, -0.0172, -0.0072, -0.9998,  1.0000,  0.0000,
          0.0000,  0.0069, -0.0102,  0.0315,  0.0103, -0.1056, -0.0014,  0.0178,
         -0.0114,  0.0365,  0.0062, -0.0995,  0.0231,  0.0397, -0.0806,  0.2872,
          0.0530, -0.8428, -0.0187,  0.0988, -0.0809,  0.3314,  0.0075, -0.8272,
          0.0787, -0.0109,  0.2395, -0.3302, -0.0538, -0.6919, -0.0286,  0.0652,
          0.6694,  0.1044, -0.1323, -0.9079,  0.0219]], device='cuda:0')
torques: [  -5.61468918   57.82955759  200.         -200.         -200.
   33.55022055    6.86344991 -118.70694782  160.16736778 -153.73908733
 -200.          -36.93369555]
dof_po

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)

    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[ 0.0318, -0.7023, -0.6884, -0.7177,  0.7075,  0.0979, -0.5392, -0.3198,
         -0.7516, -0.9758,  0.3635, -0.0685]], device='cuda:0')
Scaled actions :  tensor([[ 0.0318, -0.7023, -0.6884, -0.7177,  0.7075,  0.0979, -0.5392, -0.3198,
         -0.7516, -0.9758,  0.3635, -0.0685]], device='cuda:0')
obs :  tensor([[-3.1030e-01, -1.6413e-01, -1.5852e-01, -3.4143e-02, -7.0578e-03,
         -9.9939e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  5.7116e-03,
         -2.8668e-04,  7.2451e-02,  1.4917e-02, -2.9818e-01, -1.0081e-02,
          3.6122e-02, -2.2854e-03,  8.6771e-02,  1.2573e-02, -3.3911e-01,
          1.9384e-02, -2.6622e-02,  1.5600e-01,  1.2211e-01, -2.2019e-02,
         -8.3533e-01, -4.2608e-02,  7.6329e-02,  1.4537e-01,  1.6173e-01,
          3.4472e-02, -1.2010e+00, -2.6574e-02,  3.1831e-02, -7.0232e-01,
         -6.8839e-01, -7.1771e-01,  7.0746e-01,  9.7927e-02, -5.3919e-01,
         -3.1980e-01, -7.5162e-01, -9.7575e-01,  3.6347e-01, -6.8

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)

    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[ 4.4680e-02, -1.3449e-01,  1.4352e+00,  2.4564e-01,  5.3759e-01,
          7.5959e-02,  9.1448e-02,  9.3689e-01,  5.9528e-01,  3.7880e-04,
          2.3236e-01,  2.9301e-01]], device='cuda:0')
Scaled actions :  tensor([[ 4.4680e-02, -1.3449e-01,  1.4352e+00,  2.4564e-01,  5.3759e-01,
          7.5959e-02,  9.1448e-02,  9.3689e-01,  5.9528e-01,  3.7880e-04,
          2.3236e-01,  2.9301e-01]], device='cuda:0')
obs :  tensor([[ 3.3873e-01,  5.0866e-01,  1.8840e-01, -2.8227e-02, -6.2810e-03,
         -9.9958e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.0654e-02,
         -2.9423e-03,  6.7888e-02,  1.1519e-02, -3.2560e-01,  2.0601e-02,
         -1.3587e-02, -3.8652e-03,  9.3873e-02,  3.8153e-03, -4.4531e-01,
         -1.1138e-02,  2.6376e-02, -1.3804e-01, -1.3875e-01, -2.3066e-02,
          4.1340e-01,  1.6534e-01, -4.8571e-01, -1.4848e-01, -9.3712e-02,
         -4.0199e-02,  2.4585e-02, -1.1105e-01,  4.4680e-02, -1.3449e-01,
          1.4352e+00,  2.

In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)

    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[-0.1929,  0.2890,  0.0122,  0.0673, -0.4593,  0.0353,  1.1137,  0.3443,
          0.2529, -0.1546,  0.2348, -0.2545]], device='cuda:0')
Scaled actions :  tensor([[-0.1929,  0.2890,  0.0122,  0.0673, -0.4593,  0.0353,  1.1137,  0.3443,
          0.2529, -0.1546,  0.2348, -0.2545]], device='cuda:0')
obs :  tensor([[ 0.1970, -0.1922, -0.1515, -0.0214, -0.0183, -0.9996,  1.0000,  0.0000,
          0.0000,  0.0236, -0.0290,  0.0601,  0.0204, -0.1190,  0.0390, -0.0430,
         -0.0244,  0.1038, -0.0065, -0.3123,  0.0657,  0.0700, -0.1184,  0.0451,
          0.0975,  1.3766,  0.0801,  0.1245, -0.0596,  0.1687, -0.0551,  1.1135,
          0.4864, -0.1929,  0.2890,  0.0122,  0.0673, -0.4593,  0.0353,  1.1137,
          0.3443,  0.2529, -0.1546,  0.2348, -0.2545]], device='cuda:0')
torques: [200.         200.         200.          65.90719611  45.75738664
 -31.36679653 -25.64883276 -94.55471065 200.         200.
 -27.12757989  -5.77235078]
dof_pos: [ 0.02355

In [26]:
# 既存のforループを置き換え
num_steps = 100
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        dof_pos = env.dof_pos[0].cpu().numpy()
        dof_vel = env.dof_vel[0].cpu().numpy()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        dof_pos_data.append(dof_pos)
        dof_vel_data.append(dof_vel)
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=2.245, Scaled action max=2.245
Step 1/100, Total steps: 806
steps: 806
actions : tensor([[-0.4444,  0.3151,  0.4235,  0.1269, -1.1269,  0.1391,  0.3426, -1.2396,
          1.0635,  0.9381,  2.2452, -0.8132]], device='cuda:0')
target_dof_pos: tensor([[ 0.2359, -0.3282, -0.8476,  1.5498, -1.1802, -0.2080,  0.2279, -1.1280,
         -0.3565,  0.8122,  0.9358, -0.2613]], device='cuda:0')
Step 1: Original action max=2.357, Scaled action max=2.357
Step 2: Original action max=2.428, Scaled action max=2.428
Step 21/100, Total steps: 826
steps: 826
actions : tensor([[-0.4139, -0.1878, -0.1002, -0.2109, -1.1225, -0.3676,  0.6369,  0.9617,
          0.1347, -1.5583, -1.0902,  0.9314]], device='cuda:0')
target_dof_pos: tensor([[ 0.0287, -0.7952, -0.9938,  1.5222, -1.7952, -0.3207, -0.1646,  1.3441,
         -1.1891,  0.3310, -2.6973,  1.2323]], device='cuda:0')
Step 41/100, Total steps: 846
steps: 846
actions : tensor([[-0.1251, -0.4525,  0.7148, -0.5861, -1.1839, -0.53

In [19]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [27]:
env.sim.stop()

In [21]:
env.reset()
cnt = 0

In [20]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]

    # dof_posデータ
    dof_pos_array = np.array(dof_pos_data)
    for i in range(dof_pos_array.shape[1]):
        data_dict[f'dof_pos_{i}'] = dof_pos_array[:, i]

    # dof_velデータ
    dof_vel_array = np.array(dof_vel_data)
    for i in range(dof_vel_array.shape[1]):
        data_dict[f'dof_vel_{i}'] = dof_vel_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.1.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-terrain1-kp2000kd50-kpkdrand-26-norand_ckpt100_scale1.0_rotorInertia0.1.csv
データ形状: (501, 82)
